# Check interpolation inputs

Audits the Ti_Q_2A `H(R)`/MMN-derived `A(R)` interpolation inputs -- the layer immediately upstream of the Berry-curvature decomposition, checked without using it:

- `H(R)` against the checkpoint Hamiltonian on the original $8^3$ mesh;
- exact inverse-FFT reconstruction of the MMN finite-difference $A(k)$;
- invariance of mesh-node values under the orbital-aware R remapping;
- the production Hermitian projection and on-disk `A(R)` cache;
- Hermiticity and convention-I reciprocal covariance off the mesh; and
- analytic $dH$ and $\mathrm{curl}(A)$ against central differences of the same interpolants.

The large shared MMN is streamed once per checkpoint. No production file is modified. `RUN_COMPUTATION = False` loads the last saved `results/interpolation_inputs/interpolation_checks_summary.json` instead of rerunning (this is the slowest check in `inputs/` -- it streams the full MMN).

In [ ]:
import sys
import json
import time
import os
from pathlib import Path
from typing import Mapping

import numpy as np
from pythtb import W90

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from validation._paths import (
    RESULTS,
    STO_AA_CACHE,
    STO_TI_Q_2A_RUN,
)
from validation._paths import DATA
from modules import curvature as cv
from wannier.wannier_io import _nnkp_block, read_wannier_checkpoint
from wannier.position_matrix import (
    _connection_to_A_R,
    _map_R_by_orbital_positions,
    apply_orbital_dependent_R_mapping,
    build_position_matrix_from_mmn,
    connection_from_mmn,
)

## Config

In [ ]:
RUN_COMPUTATION = False  # True: rebuild and recompute (streams the full MMN, slow). False: load results/interpolation_inputs/.
PREFIX = "SrTiO3"
RUN = STO_TI_Q_2A_RUN
MMN_DONOR = RUN / "trial_03_Sr_sp_Ti_pd_O_sp/proj_gauge"
CACHE_DIR = STO_AA_CACHE
OUTPUT_DIR = RESULTS / "interpolation_inputs"
OUTPUT_PATH = OUTPUT_DIR / "interpolation_checks_summary.json"

TRIALS = {
    "trial03": {
        "directory": MMN_DONOR,
        "cache": CACHE_DIR / "AA_Ti_Q_2A_trial03_48wf_mmn_pair_ws1em05.npz",
    },
    "trial04": {
        "directory": DATA / "SrTiO3/Ti_Q_2A/output/188914bc889/trial_04_Sr_sp_Ti_p_O_sp/proj_gauge/188927bc889",
        "cache": CACHE_DIR / "AA_Ti_Q_2A_trial04_38wf_mmn_pair_ws1em05.npz",
    },
}
TRIALS_TO_RUN = list(TRIALS)  # was --trial: "trial03", "trial04", or both (default)
PLANES = ((1, 2), (2, 0), (0, 1))
R_TOLERANCE = 1.0e-5

## Small numeric helpers

In [ ]:
def max_abs(value: np.ndarray) -> float:
    return float(np.max(np.abs(value)))


def relative_error(actual: np.ndarray, expected: np.ndarray) -> float:
    denominator = max(float(np.linalg.norm(expected)), 1.0e-300)
    return float(np.linalg.norm(actual - expected) / denominator)


def metrics(actual: np.ndarray, expected: np.ndarray) -> dict[str, float]:
    return {
        "max_abs": max_abs(actual - expected),
        "relative_frobenius": relative_error(actual, expected),
    }


def hermiticity_metrics(matrices: np.ndarray) -> dict[str, float]:
    defect = matrices - matrices.conj().swapaxes(-1, -2)
    return {
        "max_abs": max_abs(defect),
        "relative_frobenius": float(
            np.linalg.norm(defect) / max(np.linalg.norm(matrices), 1.0e-300)
        ),
    }


def convention_i_transform(field: np.ndarray, phases: np.ndarray) -> np.ndarray:
    """D^dag field D, with the last two axes carrying orbital indices."""
    return np.conj(phases)[..., :, None] * field * phases[..., None, :]

## R <-> k plumbing (folding, inverse FFT, plain Bloch sums, checkpoint ordering)

In [ ]:
def fold_blocks(blocks: Mapping[tuple[int, int, int], np.ndarray], mp_grid: tuple[int, int, int]) -> np.ndarray:
    """Fold arbitrary R representatives into one discrete Fourier cell."""
    sample = np.asarray(next(iter(blocks.values())))
    folded = np.zeros(mp_grid + sample.shape, dtype=complex)
    modulus = np.asarray(mp_grid, dtype=int)
    for vector, block in blocks.items():
        folded[tuple(np.mod(vector, modulus))] += block
    return folded


def grid_values_from_R(folded_R: np.ndarray) -> np.ndarray:
    """Return values on an ordered FFT mesh for the +2pi i k.R convention."""
    return np.fft.ifftn(folded_R, axes=(0, 1, 2)) * np.prod(folded_R.shape[:3])


def bloch_sum_R(blocks: Mapping[tuple[int, int, int], np.ndarray], kpoints: np.ndarray) -> np.ndarray:
    """Evaluate a small batch of plain +2pi i k.R Bloch sums."""
    vectors = np.asarray(list(blocks), dtype=float)
    values = np.asarray([blocks[R] for R in blocks])
    phases = np.exp(2j * np.pi * (np.asarray(kpoints) @ vectors.T))
    return np.tensordot(phases, values, axes=(1, 0))


def checkpoint_order(grid: np.ndarray, checkpoint: dict) -> np.ndarray:
    mp_grid = np.asarray(checkpoint["mp_grid"], dtype=int)
    coordinates = np.rint(np.asarray(checkpoint["kpt_red"], dtype=float) * mp_grid[None]).astype(int)
    coordinates %= mp_grid
    return np.asarray([grid[tuple(point)] for point in coordinates])


def checkpoint_eigenvalues(directory: Path, checkpoint: dict) -> np.ndarray:
    raw = np.loadtxt(directory / f"{PREFIX}.eig")
    result = np.empty((int(checkpoint["num_kpts"]), int(checkpoint["num_bands"])), dtype=float)
    result[raw[:, 1].astype(int) - 1, raw[:, 0].astype(int) - 1] = raw[:, 2]
    return result


def read_connection(checkpoint: dict, directory: Path) -> dict:
    nnkp = directory / f"{PREFIX}.nnkp"
    reciprocal_lattice = None
    if nnkp.is_file():
        values = np.fromstring(_nnkp_block(nnkp, "recip_lattice"), sep=" ")
        if values.size != 9:
            raise ValueError(f"Expected a 3x3 reciprocal lattice in {nnkp}")
        reciprocal_lattice = values.reshape(3, 3)
    return connection_from_mmn(
        checkpoint, MMN_DONOR / f"{PREFIX}.mmn",
        translational_invariant_MV=False, reciprocal_lattice=reciprocal_lattice,
    )

## H(R) checks

In [ ]:
def hamiltonian_mesh_checks(w90: W90, checkpoint: dict, directory: Path) -> dict:
    mp_grid = tuple(int(value) for value in checkpoint["mp_grid"])
    raw = {
        R: np.asarray(block["h"], dtype=complex) / float(block["deg"])
        for R, block in w90.ham_r.items()
    }
    mapped = _map_R_by_orbital_positions(
        raw, np.asarray(w90.lat, dtype=float),
        np.asarray(checkpoint["wannier_centres"], dtype=float), mp_grid,
        tolerance=R_TOLERANCE,
    )
    raw_folded = fold_blocks(raw, mp_grid)
    mapped_folded = fold_blocks(mapped, mp_grid)
    raw_grid = grid_values_from_R(raw_folded)
    raw_nodes = checkpoint_order(raw_grid, checkpoint)

    eigenvalues = checkpoint_eigenvalues(directory, checkpoint)
    V = np.asarray(checkpoint["v_matrix"])
    checkpoint_H = np.einsum("kbi,kb,kbj->kij", V.conj(), eigenvalues, V, optimize=True)

    mapped_weight = sum(
        float(np.vdot(block, block).real) for R, block in mapped.items() if R not in raw
    )
    total_weight = sum(float(np.vdot(block, block).real) for block in mapped.values())
    diagnostic_k = np.random.default_rng(7331).uniform(-0.2, 1.2, size=(5, 3))
    return {
        "n_R_raw": len(raw),
        "n_R_mapped": len(mapped),
        "weight_moved_outside_raw_R_sqrt_fraction": float(
            np.sqrt(mapped_weight / total_weight) if total_weight else 0.0
        ),
        "raw_vs_mapped_folded_coefficients": metrics(mapped_folded, raw_folded),
        "raw_vs_mapped_original_mesh_values": metrics(grid_values_from_R(mapped_folded), raw_grid),
        "raw_vs_mapped_offgrid_values_expected_to_differ": metrics(
            bloch_sum_R(mapped, diagnostic_k), bloch_sum_R(raw, diagnostic_k)
        ),
        "hr_vs_checkpoint_H_original_mesh": metrics(raw_nodes, checkpoint_H),
        "hr_vs_checkpoint_eigenvalues_max_abs_eV": float(
            np.max(np.abs(np.linalg.eigvalsh(raw_nodes) - np.linalg.eigvalsh(checkpoint_H)))
        ),
        "raw_mesh_H_hermiticity": hermiticity_metrics(raw_nodes),
    }

## A(R) checks (the MMN -> production connection pipeline)

In [ ]:
def connection_mesh_checks(raw_A_k: np.ndarray, checkpoint: dict, cache_path: Path) -> dict:
    """Check every transformation used by MMN -> production A(R)."""
    mp_grid = tuple(int(value) for value in checkpoint["mp_grid"])
    num_wann = int(checkpoint["num_wann"])
    coordinates = np.rint(np.asarray(checkpoint["kpt_red"]) * np.asarray(mp_grid)[None]).astype(int)
    coordinates %= np.asarray(mp_grid)

    A_grid = np.zeros(mp_grid + (num_wann, num_wann, 3), dtype=complex)
    for index, point in enumerate(coordinates):
        A_grid[tuple(point)] = raw_A_k[index]
    raw_R_grid = np.fft.fftn(A_grid, axes=(0, 1, 2)) / np.prod(mp_grid)
    raw_R = {R: raw_R_grid[R].transpose(2, 0, 1) for R in np.ndindex(mp_grid)}
    mapped_prehermitian = _map_R_by_orbital_positions(
        raw_R, np.asarray(checkpoint["real_lattice"], dtype=float),
        np.asarray(checkpoint["wannier_centres"], dtype=float), mp_grid,
        tolerance=R_TOLERANCE,
    )
    raw_folded = fold_blocks(raw_R, mp_grid)
    mapped_folded = fold_blocks(mapped_prehermitian, mp_grid)

    reconstructed_grid = grid_values_from_R(raw_R_grid)
    reconstructed_k = checkpoint_order(reconstructed_grid, checkpoint)
    mapped_grid = grid_values_from_R(mapped_folded).transpose(0, 1, 2, 4, 5, 3)
    mapped_k = checkpoint_order(mapped_grid, checkpoint)

    production_A_R, production_R = _connection_to_A_R(checkpoint, raw_A_k.copy(), R_TOLERANCE)
    production_blocks = {
        tuple(int(value) for value in vector): block.transpose(2, 0, 1)
        for vector, block in zip(production_R, production_A_R)
    }
    production_grid = grid_values_from_R(fold_blocks(production_blocks, mp_grid))
    production_grid = production_grid.transpose(0, 1, 2, 4, 5, 3)
    production_k = checkpoint_order(production_grid, checkpoint)
    hermitian_raw = 0.5 * (raw_A_k + raw_A_k.conj().swapaxes(1, 2))

    with np.load(cache_path, allow_pickle=False) as archive:
        cached_A_R = np.asarray(archive["AA"])
        cached_R = np.asarray(archive["iRvec"])
    cache_same_R = np.array_equal(cached_R, production_R)
    cache_error = (
        metrics(cached_A_R, production_A_R) if cache_same_R
        else {"max_abs": float("inf"), "relative_frobenius": float("inf")}
    )

    origin = np.flatnonzero(np.all(production_R == 0, axis=1))
    if len(origin) != 1:
        raise ValueError("Production A(R) does not have exactly one origin")
    diagonal = np.arange(num_wann)
    origin_diagonal = production_A_R[origin[0], diagonal, diagonal]
    diagnostic_k = np.random.default_rng(188725).uniform(-0.2, 1.2, size=(5, 3))

    return {
        "inverse_fft_raw_A_original_mesh": metrics(reconstructed_k, raw_A_k),
        "raw_vs_mapped_folded_coefficients": metrics(mapped_folded, raw_folded),
        "raw_vs_mapped_original_mesh_values": metrics(mapped_k, raw_A_k),
        "raw_vs_mapped_offgrid_values_expected_to_differ": metrics(
            bloch_sum_R(mapped_prehermitian, diagnostic_k).transpose(0, 2, 3, 1),
            bloch_sum_R(raw_R, diagnostic_k).transpose(0, 2, 3, 1),
        ),
        "raw_A_mesh_hermiticity": hermiticity_metrics(raw_A_k.transpose(0, 3, 1, 2)),
        "production_A_mesh_vs_hermitian_part_of_raw": metrics(production_k, hermitian_raw),
        "production_projection_change_from_raw": metrics(production_k, raw_A_k),
        "production_R0_diagonal_max_abs_Angstrom": max_abs(origin_diagonal),
        "cache_R_vectors_identical": bool(cache_same_R),
        "regenerated_vs_cached_A_R": cache_error,
        "n_R_raw": len(raw_R),
        "n_R_mapped_prehermitian": len(mapped_prehermitian),
        "n_R_production": len(production_R),
    }

## Central-difference and off-grid structure checks

`cv.fields(model, pos, kpoints)` returns `.A` and `.curl` (direction first, `(3, nk, J, J)`; the checks below use the k-first view from `bloch_fields`) -- inspect those directly if you run these functions interactively at your own k-points.

In [ ]:
def bloch_fields(model, pos, kpoints):
    """A(k) and its plain curl as (nk, 3, J, J): k first, the layout the checks below index."""
    f = cv.fields(model, pos, kpoints)
    return f.A.swapaxes(0, 1), f.curl.swapaxes(0, 1)


def central_difference_checks(model, pos: cv.PositionTerms) -> dict:
    rng = np.random.default_rng(240827)
    kpoints = rng.uniform(-0.35, 1.35, size=(4, 3))
    analytic_dH = np.asarray(model.velocity(kpoints, flatten_spin_axis=True), dtype=np.complex128)
    analytic_A, analytic_curl = bloch_fields(model, pos, kpoints)
    analytic_curl = np.asarray(analytic_curl, dtype=np.complex128)
    steps = (1.0e-2, 3.0e-3, 1.0e-3, 3.0e-4, 1.0e-4, 3.0e-5, 1.0e-5)
    dH_results: dict[str, dict[str, float]] = {}
    curl_results: dict[str, dict[str, float]] = {}
    for step in steps:
        numerical_dH = np.empty_like(analytic_dH)
        derivatives_A = np.empty((3,) + analytic_A.shape, dtype=np.complex128)
        for axis in range(3):
            displacement = np.zeros(3)
            displacement[axis] = step
            plus = kpoints + displacement
            minus = kpoints - displacement
            numerical_dH[axis] = (
                model.hamiltonian(plus, flatten_spin_axis=True)
                - model.hamiltonian(minus, flatten_spin_axis=True)
            ) / (2.0 * step)
            derivatives_A[axis] = (
                bloch_fields(model, pos, plus)[0] - bloch_fields(model, pos, minus)[0]
            ) / (2.0 * step)
        numerical_curl = np.stack(
            [derivatives_A[mu, :, nu] - derivatives_A[nu, :, mu] for mu, nu in PLANES], axis=1,
        )
        key = f"{step:.0e}"
        dH_results[key] = metrics(numerical_dH, analytic_dH)
        curl_results[key] = metrics(numerical_curl, analytic_curl)

    best_dH = min(dH_results.items(), key=lambda item: item[1]["relative_frobenius"])
    best_curl = min(curl_results.items(), key=lambda item: item[1]["relative_frobenius"])
    return {
        "n_random_kpoints": len(kpoints),
        "central_difference_steps_reduced": list(steps),
        "dH_by_step": dH_results,
        "curl_by_step": curl_results,
        "best_dH": {"step": best_dH[0], **best_dH[1]},
        "best_curl": {"step": best_curl[0], **best_curl[1]},
    }


def offgrid_structure_checks(model, pos: cv.PositionTerms) -> dict:
    rng = np.random.default_rng(910247)
    kpoints = rng.uniform(-0.4, 1.4, size=(7, 3))
    G = np.array([1, -1, 2], dtype=int)
    shifted = kpoints + G
    phases = np.exp(2j * np.pi * (np.asarray(model.orb_vecs) @ G))

    H = np.asarray(model.hamiltonian(kpoints, flatten_spin_axis=True))
    H_shifted = np.asarray(model.hamiltonian(shifted, flatten_spin_axis=True))
    A, curl = bloch_fields(model, pos, kpoints)
    shifted_A, shifted_curl = bloch_fields(model, pos, shifted)
    expected_H = convention_i_transform(H, phases)
    expected_A = convention_i_transform(np.asarray(A), phases)
    expected_curl = convention_i_transform(np.asarray(curl), phases)
    return {
        "n_random_kpoints": len(kpoints),
        "reciprocal_shift": G.tolist(),
        "H_hermiticity": hermiticity_metrics(H),
        "A_hermiticity": hermiticity_metrics(np.asarray(A)),
        "curl_hermiticity": hermiticity_metrics(np.asarray(curl)),
        "H_reciprocal_covariance": metrics(H_shifted, expected_H),
        "A_reciprocal_covariance": metrics(np.asarray(shifted_A), expected_A),
        "curl_reciprocal_covariance": metrics(np.asarray(shifted_curl), expected_curl),
    }

## Pass/fail thresholds

In [ ]:
def thresholds_and_status(result: dict) -> tuple[dict[str, float], dict[str, bool]]:
    thresholds = {
        "fft_or_mapping_relative": 2.0e-12,
        "hr_checkpoint_max_abs_eV": 1.0e-4,
        "cache_relative": 2.0e-12,
        "hermiticity_relative": 2.0e-12,
        "reciprocal_covariance_relative": 2.0e-11,
        "derivative_relative": 2.0e-7,
    }
    H = result["hamiltonian_mesh"]
    A = result["connection_mesh"]
    off = result["offgrid_structure"]
    diff = result["central_differences"]
    status = {
        "H_R_mapping_node_invariance": H["raw_vs_mapped_original_mesh_values"]["relative_frobenius"] < thresholds["fft_or_mapping_relative"],
        "H_R_reconstructs_checkpoint": H["hr_vs_checkpoint_H_original_mesh"]["max_abs"] < thresholds["hr_checkpoint_max_abs_eV"],
        "A_inverse_fft": A["inverse_fft_raw_A_original_mesh"]["relative_frobenius"] < thresholds["fft_or_mapping_relative"],
        "A_R_mapping_node_invariance": A["raw_vs_mapped_original_mesh_values"]["relative_frobenius"] < thresholds["fft_or_mapping_relative"],
        "A_production_hermitian_projection": A["production_A_mesh_vs_hermitian_part_of_raw"]["relative_frobenius"] < thresholds["fft_or_mapping_relative"],
        "A_cache_matches_regeneration": A["cache_R_vectors_identical"] and A["regenerated_vs_cached_A_R"]["relative_frobenius"] < thresholds["cache_relative"],
        "offgrid_H_hermitian": off["H_hermiticity"]["relative_frobenius"] < thresholds["hermiticity_relative"],
        "offgrid_A_hermitian": off["A_hermiticity"]["relative_frobenius"] < thresholds["hermiticity_relative"],
        "offgrid_curl_hermitian": off["curl_hermiticity"]["relative_frobenius"] < thresholds["hermiticity_relative"],
        "H_reciprocal_covariance": off["H_reciprocal_covariance"]["relative_frobenius"] < thresholds["reciprocal_covariance_relative"],
        "A_reciprocal_covariance": off["A_reciprocal_covariance"]["relative_frobenius"] < thresholds["reciprocal_covariance_relative"],
        "curl_reciprocal_covariance": off["curl_reciprocal_covariance"]["relative_frobenius"] < thresholds["reciprocal_covariance_relative"],
        "analytic_dH_matches_central_difference": diff["best_dH"]["relative_frobenius"] < thresholds["derivative_relative"],
        "analytic_curl_matches_central_difference": diff["best_curl"]["relative_frobenius"] < thresholds["derivative_relative"],
    }
    return thresholds, {key: bool(value) for key, value in status.items()}

## Run one trial

Returns `(result, artifacts)`: `result` is the JSON-serializable metrics dict, `artifacts` holds the live objects (`checkpoint`, `w90_raw`, `model`, `pos`) so you can keep exploring after the cell runs -- e.g. `cv.fields(artifacts["model"], artifacts["pos"], my_kpoints)`.

In [ ]:
def run_trial(label: str) -> tuple[dict, dict]:
    setup = TRIALS[label]
    directory = Path(setup["directory"])
    cache_path = Path(setup["cache"])
    start = time.perf_counter()
    print(f"{label}: reading checkpoint and H(R)", flush=True)
    checkpoint = read_wannier_checkpoint(directory / f"{PREFIX}.chk")
    w90_raw = W90(str(directory), PREFIX)
    H_checks = hamiltonian_mesh_checks(w90_raw, checkpoint, directory)

    print(f"{label}: streaming shared MMN and rebuilding A(k)", flush=True)
    connection_data = read_connection(checkpoint, directory)
    A_checks = connection_mesh_checks(connection_data["A"], checkpoint, cache_path)
    A_checks["finite_difference_shell_completeness_max_abs"] = float(connection_data["completeness_error"])
    del connection_data

    print(f"{label}: building production interpolants from warm cache", flush=True)
    w90_mapped = W90(str(directory), PREFIX)
    apply_orbital_dependent_R_mapping(w90_mapped, checkpoint["mp_grid"], verbose=False)
    model = w90_mapped.model(zero_energy=0.0, min_hopping_norm=0.0)
    installation = build_position_matrix_from_mmn(
        model, directory, PREFIX, mmn_directory=MMN_DONOR, cache=cache_path,
        translational_invariant_JM=False, translational_invariant_MV=False,
        R_distance_tolerance=R_TOLERANCE,
    )
    pos = cv.position_terms(model, dtype=np.complex128)
    offgrid = offgrid_structure_checks(model, pos)
    differences = central_difference_checks(model, pos)
    result = {
        "num_wann": int(checkpoint["num_wann"]),
        "mp_grid": [int(value) for value in checkpoint["mp_grid"]],
        "hamiltonian_mesh": H_checks,
        "connection_mesh": A_checks,
        "installed_position_matrix": {
            "n_R_installed": int(installation["n_R_pos"]),
            "n_R_in_full_MMN_transform": int(installation["n_R_mmn"]),
            "weight_discarded_outside_H_R_sqrt_fraction": float(installation["weight_outside_ham_R"]),
            "R0_diagonal_replaced_max_abs_Angstrom": float(installation["residual_R0_max"]),
            "pair_hermiticity_relative": float(installation["mmn_pair_hermiticity_relative"]),
        },
        "offgrid_structure": offgrid,
        "central_differences": differences,
        "elapsed_seconds": float(time.perf_counter() - start),
    }
    thresholds, status = thresholds_and_status(result)
    result["thresholds"] = thresholds
    result["status"] = status
    result["all_required_checks_pass"] = bool(all(status.values()))
    artifacts = {"checkpoint": checkpoint, "w90_raw": w90_raw, "model": model, "pos": pos}
    return result, artifacts

## Run + save

In [ ]:
if RUN_COMPUTATION:
    trial_results = {}
    trial_artifacts = {}
    for label in TRIALS_TO_RUN:
        trial_results[label], trial_artifacts[label] = run_trial(label)
        print(f"{label}: {'PASS' if trial_results[label]['all_required_checks_pass'] else 'FAIL'}", flush=True)
    output = {
        "description": "Ti_Q_2A exact H(R)/MMN-derived A(R) interpolation checks",
        "mmn_donor": str(MMN_DONOR / f"{PREFIX}.mmn"),
        "trials": trial_results,
        "all_required_checks_pass": bool(all(r["all_required_checks_pass"] for r in trial_results.values())),
    }
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_PATH.write_text(json.dumps(output, indent=2, sort_keys=True) + "\n")
    print(f"wrote {OUTPUT_PATH}", flush=True)

## Load a previous run (skip if you just computed above)

Only `output` (the JSON-serializable metrics) is reloadable this way -- the live `model`/`connection` objects in `trial_artifacts` only exist after `RUN_COMPUTATION = True`.

In [ ]:
if not RUN_COMPUTATION:
    output = json.loads(OUTPUT_PATH.read_text())

passed = output["all_required_checks_pass"]
for label, result in output["trials"].items():
    print(f"{label}: {'PASS' if result['all_required_checks_pass'] else 'FAIL'}")
print(f"\nall_required_checks_pass = {passed}")